# 02 - Chunking

**Goal.** Show how extracted blocks become chunks: structure-aware recursive splitting with overlap,
atomic tables, section paths and page ranges.

**What it checks** (each ends with an `assert`):
- every document is chunked with the configured size/overlap (`CHUNK_SIZE_TOKENS` / `CHUNK_OVERLAP_TOKENS`)
- no chunk exceeds the size limit, and chunk text is an exact slice of the document text (nothing lost)
- a table is kept in one chunk together with its header row; an oversized table is split by rows with the header repeated
- section paths and page ranges are carried onto the chunks
- the configured size is compared with a smaller one, by counts only

**How to run.** From the repo root: `.venv/Scripts/python.exe scripts/run_notebooks.py 02`, or open it
with a Python 3 kernel started in the repo root. No network or API key is needed.

Token counting uses `RegexTokenCounter` (words and punctuation, deterministic, no download). In production
the counter is the embedding model's own tokenizer, so real counts differ somewhat, especially for Urdu.

In [1]:
import warnings

warnings.filterwarnings("ignore")
import re
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "app").is_dir() else Path.cwd().parent
sys.path[:0] = [str(ROOT), str(ROOT / "notebooks")]

from _support import clip, make_samples, new_workdir, notebook_settings, prepare
from app.ingestion.chunking import RegexTokenCounter, chunk_blocks
from app.ingestion.cleaning import clean_blocks, join_blocks
from app.ingestion.extractors import default_extractors
from tests.fixtures.files import factories as f

settings = notebook_settings()
SIZE, OVERLAP = settings.chunk_size_tokens, settings.chunk_overlap_tokens
SMALL = 60
print(f"configured chunk size = {SIZE} tokens, overlap = {OVERLAP}; small variant = {SMALL}")
assert (SIZE, OVERLAP) == (512, 64), "defaults changed: update the notebook text"

work = new_workdir()
samples = make_samples(work)
KEYS = ["pdf_table", "pdf_pages", "docx", "pptx", "html", "md", "urdu"]

configured chunk size = 512 tokens, overlap = 64; small variant = 60


## 1. Chunk every sample with the configured size

In [2]:
prepared = {k: prepare(samples[k], settings) for k in KEYS}
print(f"{'sample':10} {'chunks':>6} {'tokens (min/avg/max)':>22} {'tables':>6}  pages")
for k, p in prepared.items():
    counts = [c.token_count for c in p.chunks]
    pages = sorted({pg for c in p.chunks for pg in (c.page_start, c.page_end) if pg})
    print(f"{k:10} {len(p.chunks):>6} {min(counts):>8}/{sum(counts) // len(counts):>5}/{max(counts):>5} "
          f"{sum(c.is_table for c in p.chunks):>6}  {pages or '-'}")
    assert p.chunks and max(counts) <= SIZE
    assert [c.index for c in p.chunks] == list(range(len(p.chunks)))

sample     chunks   tokens (min/avg/max) tables  pages
pdf_table       2       19/   29/   40      1  [1]
pdf_pages       1      170/  170/  170      0  [1, 2]
docx            6        3/   57/   83      1  -
pptx            3        1/   12/   20      1  [1, 2]
html            4        4/   20/   31      1  -
md              3        3/   18/   33      0  -
urdu            1       51/   51/   51      0  -


## 2. Chunk text is an exact slice of the document (nothing lost)
Text chunks carry `char_start`/`char_end` into the joined document text.

In [3]:
for k, p in prepared.items():
    for c in p.chunks:
        if not c.is_table:
            assert p.full_text[c.char_start : c.char_end] == c.text, (k, c.index)
    covered = sum(c.char_end - c.char_start for c in p.chunks)
    print(f"{k:10} chunks cover {covered} of {len(p.full_text)} characters")
    assert covered >= len(p.full_text) - 2 * len(p.blocks)  # only the "\n\n" joins are not inside a chunk

pdf_table  chunks cover 180 of 182 characters
pdf_pages  chunks cover 912 of 912 characters
docx       chunks cover 1778 of 1788 characters
pptx       chunks cover 139 of 143 characters
html       chunks cover 387 of 393 characters
md         chunks cover 312 of 316 characters
urdu       chunks cover 243 of 243 characters


## 3. Section paths and page ranges
A chunk never crosses a section boundary, so each chunk has one section path (for DOCX/HTML/PPTX)
and the page range of the blocks it contains (for PDF/PPTX).

In [4]:
p = prepared["docx"]
print(f"{'#':>2} {'tokens':>6} {'table':5}  section path")
for c in p.chunks:
    print(f"{c.index:>2} {c.token_count:>6} {str(c.is_table):5}  {' > '.join(c.section_path)}  | {clip(c.text, 40)!r}")
assert len({c.section_path for c in p.chunks}) >= 4, "sections should not be merged into one chunk"
assert all(c.section_path[0] == "NADRA CNIC Guide" for c in p.chunks)

p = prepared["pptx"]
print("\nPPTX slides as pages:")
for c in p.chunks:
    print(f"  chunk {c.index}: slide {c.page_start}-{c.page_end}  {' > '.join(c.section_path)}")
assert {c.page_start for c in p.chunks} == {1, 2}

# a two-page PDF at the small size, to see a chunk range that spans pages
pdf_small = prepare(samples["pdf_pages"], settings, chunk_size=SMALL)
for c in pdf_small.chunks:
    print(f"pdf chunk {c.index}: pages {c.page_start}-{c.page_end}, {c.token_count} tokens")
assert {c.page_start for c in pdf_small.chunks} == {1, 2}
assert pdf_small.chunks[0].page_start == 1 and pdf_small.chunks[-1].page_end == 2

 # tokens table  section path
 0      3 False  NADRA CNIC Guide  | 'NADRA CNIC Guide'
 1     83 False  NADRA CNIC Guide > Eligibility  | 'Eligibility NADRA issues the computeris…'
 2     81 False  NADRA CNIC Guide > Renewal  | 'Renewal A citizen can renew the identit…'
 3     50 False  NADRA CNIC Guide > Fees  | 'Fees The fee for the normal renewal cat…'
 4     77 False  NADRA CNIC Guide > Documents required  | 'Documents required The applicant must b…'
 5     52 True   NADRA CNIC Guide > Documents required  | '| Category | Processing time | Fee | |-…'

PPTX slides as pages:
  chunk 0: slide 1-1  FBR income tax filing
  chunk 1: slide 2-2  Fees
  chunk 2: slide 2-2  Fees
pdf chunk 0: pages 1-1, 47 tokens
pdf chunk 1: pages 1-1, 46 tokens
pdf chunk 2: pages 1-2, 59 tokens
pdf chunk 3: pages 1-2, 51 tokens
pdf chunk 4: pages 2-2, 52 tokens
pdf chunk 5: pages 2-2, 43 tokens
pdf chunk 6: pages 2-2, 51 tokens


## 4. Tables stay atomic with their header
A table that fits is one chunk: the header row and every data row together, so a fee never loses its
column name.

In [5]:
p = prepared["pdf_table"]
table_chunks = [c for c in p.chunks if c.is_table]
print(table_chunks[0].text)
assert len(table_chunks) == 1
assert table_chunks[0].text.splitlines()[0] == "| Category | Pages | Fee |"
assert "Rs 4,500" in table_chunks[0].text and "Rs 9,000" in table_chunks[0].text

# An oversized table is split by rows and the header (and separator) row is repeated in every part.
big_rows = [["Category", "Office", "Fee"]] + [[f"Category {i}", f"Regional office {i}", f"Rs {1000 + i}"] for i in range(40)]
big = work / "big_table.docx"
f.make_docx(big, title="Big fee table", sections=[("Fees", ())], table=big_rows)
extracted = default_extractors()["docx"].extract(big)
blocks = clean_blocks(extracted.blocks)
parts = chunk_blocks(blocks, counter=RegexTokenCounter(), chunk_size=SMALL, overlap=8)
table_parts = [c for c in parts if c.is_table]
print(f"\n40-row table at size {SMALL}: {len(table_parts)} table chunks")
for c in table_parts[:3]:
    lines = c.text.splitlines()
    print(f"  {c.token_count:>3} tokens, {len(lines):>2} lines, starts with {lines[0]!r}")
assert len(table_parts) > 1
assert all(c.text.startswith("| Category | Office | Fee |\n|---|---|---|") for c in table_parts), "header repeated"
assert all(c.token_count <= SMALL for c in table_parts)
rows_seen = [row for c in table_parts for row in re.findall(r"\| Category (\d+) \|", c.text)]
assert sorted(map(int, rows_seen)) == list(range(40)), "every data row appears exactly once"

| Category | Pages | Fee |
|---|---|---|
| Normal | 36 | Rs 4,500 |
| Urgent | 36 | Rs 9,000 |



40-row table at size 60: 14 table chunks
   53 tokens,  5 lines, starts with '| Category | Office | Fee |'
   53 tokens,  5 lines, starts with '| Category | Office | Fee |'
   53 tokens,  5 lines, starts with '| Category | Office | Fee |'


## 5. Overlap and the token-size distribution
Within one section, a long run of text is packed up to the limit, and the next chunk starts a few
tokens before the previous one ended. The sample is long enough only when the limit is small,
so the distribution is shown for the small size.

In [6]:
long_doc = work / "long.txt"
paragraph = f.ENGLISH_PARAGRAPH
long_doc.write_text("\n\n".join(f"{paragraph} Item {i}." for i in range(30)), encoding="utf-8")
extracted = default_extractors()["txt"].extract(long_doc)
blocks = clean_blocks(extracted.blocks)
chunks = chunk_blocks(blocks, counter=RegexTokenCounter(), chunk_size=SMALL, overlap=10)
full = join_blocks(blocks)
overlaps = [max(0, a.char_end - b.char_start) for a, b in zip(chunks, chunks[1:])]
print(f"{len(chunks)} chunks; overlap (characters) between neighbours: {overlaps[:8]} ...")
assert all(c.token_count <= SMALL for c in chunks)
assert all(full[c.char_start : c.char_end] == c.text for c in chunks)
assert all(a.char_start < b.char_start for a, b in zip(chunks, chunks[1:])), "positions are monotonic"
assert sum(o > 0 for o in overlaps) >= len(overlaps) - 1, "neighbouring chunks overlap"


def histogram(token_counts: list[int], size: int, bins: int = 5) -> None:
    step = max(1, size // bins)
    print(f"{'tokens':>10} {'chunks':>6}")
    for lo in range(0, size, step):
        n = sum(lo < t <= lo + step for t in token_counts)
        print(f"{lo + 1:>4}-{lo + step:<5} {n:>6}  {'#' * n}")


histogram([c.token_count for c in chunks], SMALL)

40 chunks; overlap (characters) between neighbours: [48, 54, 57, 61, 48, 54, 57, 61] ...
    tokens chunks
   1-12         0  
  13-24         0  
  25-36         1  #
  37-48        28  ############################
  49-60        11  ###########


## 6. Compare two chunk sizes (counts only)
A smaller size gives more, shorter chunks (finer retrieval, more vectors to store and embed);
a larger one gives fewer, broader chunks.

In [7]:
print(f"{'sample':10} {'size ' + str(SIZE):>10} {'size ' + str(SMALL):>9}   chunk-count ratio")
total_big = total_small = 0
for k in KEYS:
    big_n = len(prepare(samples[k], settings, chunk_size=SIZE).chunks)
    small_n = len(prepare(samples[k], settings, chunk_size=SMALL, overlap=8).chunks)
    total_big += big_n
    total_small += small_n
    print(f"{k:10} {big_n:>10} {small_n:>9}   x{small_n / big_n:.1f}")
    assert small_n >= big_n, k
print(f"{'total':10} {total_big:>10} {total_small:>9}")
assert total_small > total_big
print("\nAll chunking checks passed.")

sample       size 512   size 60   chunk-count ratio


pdf_table           2         2   x1.0
pdf_pages           1         4   x4.0


docx                6         9   x1.5
pptx                3         3   x1.0
html                4         4   x1.0
md                  3         3   x1.0
urdu                1         1   x1.0
total              20        26

All chunking checks passed.
